# 07 — M9 variants, learned rerankers and deployment selection

Compare M8, M9 without region, M9 + region, Logistic Regression, Random Forest and XGBoost. M5 is an additional full-catalog test comparator. M9 + region extends the already-completed regional experiment; the nonregional variant isolates the change in region policy. Do not select from the test leaderboard.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
PROJECT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'recommender').is_dir()), None)
assert PROJECT is not None, 'Open within the project folder'
import os
RESULTS = Path(os.environ.get('RESULTS_DIR', str(PROJECT / 'results/final'))).resolve()


## Reproduce only when needed

Existing complete results are loaded by default. To run from scratch, use the pipeline command with a fresh run directory. The offline command is also documented in README.md. Training applicants exclude both development and test applicants. All feature extraction and candidate generation are shared with online inference.


In [2]:
# Training is deliberately not repeated in this report notebook.
# Run `python -m recommender.pipeline run --run-dir artifacts/runs/my-run` once.
assert (RESULTS / 'selection.json').exists(), 'Run the pipeline first.'


## Training and selection

The three learned models have fixed documented hyperparameters. Their labels are recorded invitation title/region groups; other retrieved groups are weak negatives. Keep all retrieved positives and sample up to 30 other candidates per training applicant, with equal total sample weight per applicant. There is no gold injection into candidate pools. M9 weights are tuned on development grouped Recall@10 only. The deployment policy retains M8 unless a candidate has a positive lower paired-development bootstrap bound after correction across five candidates; eligible candidates are ordered by development Recall@10. This conservative safeguard was added after exploratory comparisons and is not preregistered. Test outcomes never enter fitting or selection.


In [3]:
selection = json.loads((RESULTS / 'selection.json').read_text())
display(selection)
display(pd.read_csv(RESULTS / 'development_skill_weights.csv'))
display(pd.read_csv(RESULTS / 'development_metrics_grouped.csv'))
display(pd.read_csv(RESULTS / 'development_promotion.csv'))


{'selected_model': 'M8',
 'criterion': 'retain M8 unless a candidate has positive lower paired-development bootstrap bound after Bonferroni family correction; then maximize development grouped Recall@10, with fixed-order ties',
 'development_max_without_promotion_gate': 'M9 + region',
 'promotion_protocol': '10,000 paired bootstrap resamples, seed 42; 95% family coverage across five candidate comparisons (99% per comparison). Added after exploratory analysis; not preregistered.',
 'model_order': ['M8',
  'M9 without region',
  'M9 + region',
  'Logistic Regression',
  'Random Forest',
  'XGBoost'],
 'specification': {'kind': 'M8', 'use_region': True},
 'development_scores': {'M8': 0.4425,
  'M9 without region': 0.1135,
  'M9 + region': 0.4445,
  'Logistic Regression': 0.4345,
  'Random Forest': 0.44,
  'XGBoost': 0.4395},
 'training_applicants': 10227,
 'development_applicants': 1000,
 'test_applicants': 2806,
 'features': ['tfidf_cosine',
  'semantic_cosine',
  'tfidf_z',
  'semantic_

,model,skill_weight,grouped_Recall@10
0,M9 without region,0.00,0.1075
1,M9 without region,0.10,0.1055
2,M9 without region,0.25,0.1065
3,M9 without region,0.50,0.1075
4,M9 without region,1.00,0.1125
5,M9 without region,2.00,0.1135
6,M9 + region,0.00,0.4425
7,M9 + region,0.10,0.4425
8,M9 + region,0.25,0.4425
9,M9 + region,0.50,0.4435


,model,Recall@5,Precision@5,NDCG@5,Recall@10,Precision@10,NDCG@10,Recall@20,Precision@20,NDCG@20
0,M8,0.3780,0.0780,0.301320,0.4425,0.0458,0.322298,0.5270,0.02720,0.343556
1,M9 without region,0.0625,0.0128,0.037259,0.1135,0.0116,0.053647,0.1845,0.00965,0.071738
2,M9 + region,0.3755,0.0776,0.298375,0.4445,0.0460,0.320770,0.5240,0.02705,0.340846
3,Logistic Regression,0.3800,0.0786,0.307159,0.4345,0.0450,0.324893,0.5250,0.02710,0.347791
4,Random Forest,0.3835,0.0792,0.310052,0.4400,0.0455,0.328626,0.5270,0.02720,0.350606
5,XGBoost,0.3805,0.0786,0.307498,0.4395,0.0455,0.326768,0.5290,0.02730,0.349358


,model,mean_recall_gain,simultaneous_ci_lower,simultaneous_ci_upper,promote
0,M9 without region,-0.3290,-0.3680,-0.2905,False
1,M9 + region,0.0020,-0.0050,0.0100,False
2,Logistic Regression,-0.0080,-0.0220,0.0060,False
3,Random Forest,-0.0025,-0.0175,0.0135,False
4,XGBoost,-0.0030,-0.0160,0.0100,False


In [4]:
for mode in ['grouped', 'strict']:
    print(mode.upper())
    display(pd.read_csv(RESULTS / f'metrics_{mode}.csv'))


GROUPED


,model,Recall@5,Precision@5,NDCG@5,Recall@10,Precision@10,NDCG@10,Recall@20,Precision@20,NDCG@20
0,M8,0.397981,0.082324,0.323750,0.466031,0.048218,0.345729,0.539564,0.027869,0.364535
1,M9 without region,0.083737,0.017463,0.051559,0.131213,0.013685,0.066802,0.207015,0.010834,0.086218
2,M9 + region,0.398871,0.082466,0.323088,0.464071,0.048004,0.344149,0.537248,0.027744,0.362844
3,Logistic Regression,0.399406,0.082609,0.329580,0.465615,0.048182,0.351151,0.540990,0.027958,0.370353
4,Random Forest,0.397018,0.082038,0.328488,0.464962,0.048111,0.350573,0.538317,0.027815,0.369414
5,XGBoost,0.400048,0.082609,0.327952,0.465972,0.048218,0.349408,0.541702,0.027994,0.368803
6,M5,0.392635,0.081183,0.320108,0.455803,0.047113,0.340758,0.519933,0.026871,0.357081


STRICT


,model,Recall@5,Precision@5,NDCG@5,Recall@10,Precision@10,NDCG@10,Recall@20,Precision@20,NDCG@20
0,M8,0.202316,0.042267,0.142206,0.274941,0.028867,0.166082,0.368633,0.019369,0.189867
1,M9 without region,0.027684,0.005845,0.018307,0.044449,0.004740,0.023757,0.073540,0.003920,0.031165
2,M9 + region,0.201307,0.041981,0.140895,0.273634,0.028724,0.164694,0.366791,0.019298,0.188330
3,Logistic Regression,0.205643,0.042837,0.143711,0.284254,0.029793,0.169614,0.372790,0.019672,0.192293
4,Random Forest,0.206296,0.042979,0.144258,0.280167,0.029366,0.168724,0.370296,0.019494,0.191772
5,XGBoost,0.208019,0.043264,0.145146,0.284284,0.029758,0.170204,0.375938,0.019797,0.193456
6,M5,0.195842,0.040912,0.138146,0.266221,0.028011,0.161411,0.356088,0.018746,0.184410


## Inference bundle and deployment

`artifacts/deployment` contains the fitted vectorizer, sparse catalog matrix, normalized embeddings, ordered vacancy catalog, skill vectors/dictionary, selected estimator if needed, local encoder and integrity manifest. Applicant profiles and outcomes are excluded. The Streamlit app loads this selected bundle. Docker and EC2 packaging use an explicit source/model allowlist. See `docs/IMPLEMENTATION.md` for verified status and remaining external prerequisites.


In [5]:
BUNDLE = Path(os.environ.get('MODEL_BUNDLE', str(PROJECT / 'artifacts/deployment'))).resolve()
display(json.loads((BUNDLE / 'metadata.json').read_text()))


{'format_version': 1,
 'selected_model': 'M8',
 'kind': 'M8',
 'use_region': True,
 'features': ['tfidf_cosine',
  'semantic_cosine',
  'tfidf_z',
  'semantic_z',
  'hybrid_score',
  'skill_coverage',
  'same_region',
  'cv_skill_evidence',
  'job_skill_evidence',
  'exact_title'],
 'candidate_count': 300,
 'semantic_max_chars': 400,
 'lexical_weight': 0.5,
 'seed': 42,
 'vacancies': 43756,
 'embedding_dimension': 384,
 'encoder_revision': 'e8f8c211226b894fcb81acc59f3b34ba3efd5f42',
 'catalog_ids_sha256': '2d1812a715e05ee3e0ecd7e4f71ca8df0b765239e201ed8b3899e5b2d57a03fe',
 'dictionary_sha256': '22da6e32dc79afb5a4779dd463225670e540c4502a3ea1cf067cf8240c9d7f29',
 'encoder_cache_max_difference': 1.5273690223693848e-07,
 'versions': {'numpy': '1.26.4',
  'pandas': '2.2.2',
  'scipy': '1.15.3',
  'scikit-learn': '1.7.2',
  'xgboost': '3.4.1',
  'sentence-transformers': '5.2.0',
  'torch': '2.10.0',
  'joblib': '1.4.2',
  'streamlit': '1.45.1'}}